<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/database.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Develop Safely with Database Branching</h1>
    </div>
</div>

## What is Database Branching:

Database branching in SingleStore enables the creation of an isolated copy of an existing database—sharing its history but functioning independently—so you can insert, update, or delete data without impacting the parent database.

### Benefits:
- It allows rapid creation of isolated environments for prototyping or testing schema changes without disrupting production, and supports point-in-time recovery by branching to a specific timestamp, both improving speed and reducing risk.

- Traditional SQL systems required full volume-level clones or long database restores to enable branching or sandboxing. SingleStore integrates branching as a standard feature, enabling rapid, low-cost sandboxing and developer productivity without heavy SANs or long wait times.

### Use Cases:
##### 1. Development and Testing
- Instantly spin up isolated branches with the latest replica of the production database or at a specific point in time.
##### 2. Database Maintenance
- create a new database branch from the original database to rename an existing database.
##### 3. Performance Tuning
- You can evaluate multiple versions of your application on a branch, isolating and resolving any issues while the production database remains active.


### Architecture Diagram
TBD
TBD
TBD
TBD

#### Prerequisite: generate the Digital Marketing dataset
The Digital Marketing dataset can be generated using the website here: [digital-marketing.labs.singlestore.com](https://digital-marketing.labs.singlestore.com/)

1. Sign up for SingleStore Helios or Login to the portal if you already have account created. You can use SingleStore couments to get started with Helios platform if need more assistance. [Getting Started with SingleStore Helios](https://docs.singlestore.com/cloud/getting-started-with-singlestore-helios/)
2. Make sure your Standard or Premium cluster is running.
3. Open the website and fill in the connection form: your cluster host with `https://` in front (find it under **Connect** in the portal), username `admin`, your cluster password, and database name `martech`. Click **Connect**.
4. Click **Setup Database**. It creates the `martech` database and starts streaming shopper locations and purchases into it. Keep the **Simulator** switch on.
5. Under **Locations**, tick the cities you want. Each city adds ad campaigns, so more cities give the notebook more ads to search.
6. Wait until **Ad Campaigns** shows a number above 0, then come back here.
If the connection fails, check that your cluster's firewall allows your IP address.

The notebook only reads from `martech`. If you skip this step, it loads the same data from the dataset's public S3 bucket instead.

##### Before starting our tutorial, let's check table count for `martech` database we created earlier

In [34]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the cluster dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT table_name FROM information_schema.tables "
                    "WHERE table_schema = DATABASE() AND table_type = 'BASE TABLE' ORDER BY table_name")
        tables = [r[0] for r in cur.fetchall()]

        pt = PrettyTable(['table_name', 'singlestore'])
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM `{t}`")
            pt.add_row([t, cur.fetchone()[0]])

print(pt)

+-------------------------------+-------------+
|           table_name          | singlestore |
+-------------------------------+-------------+
|             cities            |      1      |
|           locations           |    301206   |
|         notifications         |    23336    |
|             offers            |     2000    |
|           purchases           |    12312    |
|            requests           |    75425    |
|            segments           |     1805    |
|            sessions           |      1      |
|      subscriber_segments      |     4016    |
|          subscribers          |     8500    |
| subscribers_last_notification |     1320    |
|          worldcities          |    128769   |
+-------------------------------+-------------+


### Create Database Branch
There are couple of ways to create a branch:<br>
**1.** Using SingleStore Helios portal<br>
**2.** Using SQL <br>
The following syntax creates a branch on the same cluster as the parent database:
&emsp; <br> ATTACH DATABASE `<parentDB_name>` AS `<new_branchDB_name>`

We will be following second apporach through the tutorial.

## Example Scenarios

**Scenario 1:** As a developer, you want to test a new business logic before implementing the query on a production dataset.



In [35]:
%%sql
## Creating Database Branch using SQL:

ATTACH DATABASE martech AS martech_branch;

RuntimeError: (singlestoredb.exceptions.OperationalError) 2591: Forwarding Error (node-236bba50-1abd-4f6e-98fb-e672b283c729-master-0.svc-236bba50-1abd-4f6e-98fb-e672b283c729:3306): Primary copy of database martech_branch is already attached. Detach it before trying to attach.
[SQL: ## Creating Database Branch using SQL:

ATTACH DATABASE martech AS martech_branch;]
(Background on this error at: https://sqlalche.me/e/21/e3q8)
If you need help solving this issue, send us a message: https://ploomber.io/community


You can navigate to the `requests` table within the `martech_branch` database and test your new business logic without impacting the production dataset. You can insert, update or delete rows, test the performance of your queries etc.

Once done testing, you can simply drop the branch database:

In [23]:
%%sql
DROP DATABASE martech_branch;

18 rows affected.

++
||
++
++

**Scenario 2:** Branching can use PITR and this requires the database to be deployed using an **Enterprise project.**

Think of a scenario when user accidently delete 5000 records from `requests` table. You can recover the data by simply creating a branch of your database at a point in time before the bad query was executed.

In [24]:
%%sql
## Creating Database Branch using SQL:
ATTACH DATABASE martech AS martech_branch;

1 rows affected.

++
||
++
++

In [25]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the cluster dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("USE martech_branch")
        cur.execute("SELECT DATABASE()");           db = cur.fetchone()[0]
        cur.execute("SELECT NOW()");                now = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM requests"); before = cur.fetchone()[0]
        cur.execute("DELETE FROM requests LIMIT 5000");          truncated = cur.rowcount
        conn.commit()
        cur.execute("SELECT COUNT(*) FROM requests"); after = cur.fetchone()[0]
        cur.execute("SELECT NOW()");                end = cur.fetchone()[0]

pt = PrettyTable(['Database', 'Time', 'Before delete', 'Rows deleted', 'After delete', 'End Time'])
pt.add_row([db, now, f"{before:,}", f"{truncated:,}", f"{after:,}", end])
print(pt)

+----------------+---------------------+---------------+--------------+--------------+---------------------+
|    Database    |         Time        | Before delete | Rows deleted | After delete |       End Time      |
+----------------+---------------------+---------------+--------------+--------------+---------------------+
| martech_branch | 2026-10-05 21:54:20 |     75,425    |    1,000     |    74,425    | 2026-10-05 21:54:20 |
+----------------+---------------------+---------------+--------------+--------------+---------------------+


In [26]:
%%sql
USE martech_branch;

SELECT count(*) FROM requests;

1 rows affected.

count(*)
74425


In [27]:
%%sql

ATTACH DATABASE martech_branch AS martech_branch_new AT TIME '2026-10-05 21_54_01';

1 rows affected.

++
||
++
++

Review rowcount after recovering database from previous state (PITR)

In [28]:
import singlestoredb as s2
from prettytable import PrettyTable

# Connection comes from the cluster dropdown at the top of the notebook
with s2.connect() as conn:
    with conn.cursor() as cur:
        cur.execute("USE martech_branch_new")
        cur.execute("SELECT DATABASE()");             db = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM requests"); count = cur.fetchone()[0]

pt = PrettyTable(['Database', 'Total Rows'])
pt.add_row([db, f"{count:,}"])
print(pt)

+--------------------+------------+
|      Database      | Total Rows |
+--------------------+------------+
| martech_branch_new |   75,425   |
+--------------------+------------+


To recover the data, you may query the missing rows and copy them to the production `requests` table.
<br>

**Scenario 3:** Alternatively, after creating database branch using PITR, you may drop the `martech_branch` database and then rename `martech_branch_new` as `martech_branch`. (I want to preserve `martech` database for further use)  
<br>
**Note**: when you do this you will have to reconnect this newly renamed sales database to your application.


To drop your existing `martech_branch` database:

In [29]:
%%sql
DROP DATABASE IF EXISTS martech_branch;

18 rows affected.

++
||
++
++

To rename `martech_branch_new` as `martech_branch`:

In [30]:
%%sql
DETACH DATABASE martech_branch_new;
ATTACH DATABASE martech_branch_new AS martech_branch;

1 rows affected.

1 rows affected.

++
||
++
++

In [31]:
%%sql
USE martech_branch;

SELECT count(*) FROM requests;

1 rows affected.

count(*)
75425


#### Clean up

Drop the `martech_branch` database from the cluster(the demo's martech database is left alone).

In [36]:
%%sql
DROP DATABASE IF EXISTS martech_branch;

18 rows affected.

++
||
++
++

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>